# Check mask coverage and tiled datasets against data_mapping

Verifies 1-1 mapping between slides in `data_mapping.csv` and:
- **MLflow masks** (downloaded from MLflow): tissue, epithelium
- **QC masks**: blur (`Piqe_piqe_median_activity_mask_`), folding (`FoldingFunction_folding_test_`), residual (`ResidualArtifactsAndCoverage_coverage_mask_`)
- **Slide files** listed in the `path` column: checks that every slide can be opened without error
- **Tiled datasets** (MLflow): checks `slides.parquet` ↔ `data_mapping` and `tiles.parquet` ↔ `slides.parquet` consistency

In [1]:
!python -m ensurepip --upgrade
!python -m pip install pandas mlflow>3 openslide-python tifffile matplotlib

Looking in links: /tmp/tmpxhfqrklg

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip


In [2]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from mlflow.artifacts import download_artifacts, list_artifacts
from openslide import OpenSlide
from tifffile import TiffFile

In [3]:
MLFLOW_TRACKING_URI = "http://mlflow-s3.rationai-mlflow"
os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

PROJECT_PATH = Path("/mnt/projects/mammaprint")
DATA_MAPPING_CSV = PROJECT_PATH / "data_mapping.csv"

# QC masks may live in several directories. They are treated as if everything
# lived in a single location: all listed dirs are merged together.
QC_MASKS_PATHS = [
    PROJECT_PATH / "qc_masks",
    PROJECT_PATH / "qc_masks3",
    PROJECT_PATH / "qc_masks_big_diff",
]

MLFLOW_MASKS = {
    "tissue": "mlflow-artifacts:/3/1d67612f123e46f6a3c6183257106512/artifacts/tissue_masks",
    "epithelium": "mlflow-artifacts:/3/0176fb9485de49699b1e16f362cff5fa/artifacts/epithelium_masks",
}

QC_MASK_PREFIXES = {
    "blur": "Piqe_piqe_median_activity_mask_",
    "folding": "FoldingFunction_folding_test_",
    "residual": "ResidualArtifactsAndCoverage_coverage_mask_",
}

TILED_DATASETS = {
    "mou_3_224": "mlflow-artifacts:/3/0b32eceb1c434abf94104bb2c54b792e/artifacts/mou_3_224",
    # "mou_2_224": "mlflow-artifacts:/3/1ebe489fd3ca4f5682c9a4b3d6f74620/artifacts/mou_2_224",
}

EMBEDDINGS = {
    "embedding3": "mlflow-artifacts:/3/16e8f2a772ec48d2a21dc8cf704f0e10/artifacts/embeddings",
}

# Functions

All logic is defined below as individually-callable functions. Run every cell in this section once
to define them, then use the **Calls** section at the bottom to run whichever checks you need.

For every mask source there are two modes:
- **list-only** (`ls`-equivalent): enumerate filenames without downloading (`list_mlflow_masks`,
  `list_qc_masks`, `list_embeddings`).
- **download + open**: download artifacts and open each file to verify readability
  (`download_mlflow_masks` + `open_masks`, `plot_sample_masks`).

In [4]:
# ---- B1. Data-mapping loader ----
def load_data_mapping(csv_path=DATA_MAPPING_CSV):
    """Load data_mapping.csv, derive expected .tiff names, return (df, expected_names)."""
    df = pd.read_csv(csv_path)
    df["tiff_name"] = df["path"].apply(lambda p: Path(p).with_suffix(".tiff").name)
    expected_names = set(df["tiff_name"])
    print(f"Loaded {len(df)} slides from {csv_path}")
    print(f"Columns: {df.columns.tolist()}")
    print(f"{len(expected_names)} unique slide tiff names expected")
    return df, expected_names

In [5]:
# ---- B2. Slide-open verification ----
SLIDE_SUFFIX_CANDIDATES = (".mrxs", ".tiff", ".tif")


def _slide_path_candidates(raw_path: str) -> list[Path]:
    path = Path(raw_path)
    candidates = [path]
    if not path.suffix:
        candidates.extend(Path(f"{raw_path}{suffix}") for suffix in SLIDE_SUFFIX_CANDIDATES)
    return candidates


def _resolve_slide_path(raw_path: str) -> Path:
    for candidate in _slide_path_candidates(raw_path):
        if candidate.is_file():
            return candidate
    return Path(raw_path)


def _try_open_slide(raw_path: str) -> tuple[Path, str | None]:
    slide_path = _resolve_slide_path(raw_path)
    try:
        str_path = str(slide_path)
        if str_path.lower().endswith((".ome.tiff", ".ome.tif")):
            with TiffFile(str_path) as slide:
                _ = len(slide.pages)
        else:
            with OpenSlide(str_path) as slide:
                _ = slide.dimensions
        return slide_path, None
    except Exception as e:
        return slide_path, f"{type(e).__name__}: {e}"


def check_slides_open(df: pd.DataFrame) -> pd.DataFrame:
    """Try to open every slide file in df. Returns a DataFrame of failures (empty => all OK)."""
    slide_open_rows = []
    for row in df.itertuples(index=False):
        opened_path, error = _try_open_slide(row.path)
        if error:
            slide_open_rows.append(
                {
                    "record_num": row.record_num,
                    "path": row.path,
                    "resolved_path": str(opened_path),
                    "error": error,
                }
            )

    slide_open_errors = pd.DataFrame(slide_open_rows)
    if slide_open_errors.empty:
        print(f"All {len(df)} slide files opened successfully.")
    else:
        print(f"{len(slide_open_errors)} slide files failed to open.")
    return slide_open_errors

In [6]:
# ---- B3. MLflow masks: two modes ----
def list_mlflow_masks(mlflow_masks=MLFLOW_MASKS) -> dict[str, set[str]]:
    """LIST ONLY: enumerate MLflow mask filenames via list_artifacts, without downloading."""
    mlflow_files: dict[str, set[str]] = {}
    for mask_name, uri in mlflow_masks.items():
        found = {
            Path(info.path).name
            for info in list_artifacts(artifact_uri=uri)
            if not info.is_dir
        }
        mlflow_files[mask_name] = found
        print(f"{len(found)} {mask_name} mask files listed (no download) at {uri}")
    return mlflow_files


def download_mlflow_masks(mlflow_masks=MLFLOW_MASKS) -> tuple[dict[str, Path], dict[str, set[str]]]:
    """DOWNLOAD: fetch MLflow mask dirs and enumerate them. Returns (dirs, files)."""
    mlflow_dirs: dict[str, Path] = {}
    mlflow_files: dict[str, set[str]] = {}
    for mask_name, uri in mlflow_masks.items():
        mask_dir = Path(download_artifacts(uri))
        mlflow_dirs[mask_name] = mask_dir
        found = {p.name for p in mask_dir.iterdir() if p.is_file()}
        mlflow_files[mask_name] = found
        print(f"{len(found)} {mask_name} mask files found in {mask_dir}")
    return mlflow_dirs, mlflow_files

In [7]:
# ---- B4. QC masks (local): list ----
def list_qc_masks(qc_paths=QC_MASKS_PATHS, prefixes=QC_MASK_PREFIXES) -> dict[str, set[str]]:
    """LIST ONLY: enumerate local QC mask filenames (prefix-stripped), without opening them.

    Accepts one or more directories; all are listed and merged, as if every mask lived in a
    single location. A single Path is also accepted for convenience.
    """
    if isinstance(qc_paths, (str, Path)):
        qc_paths = [qc_paths]
    qc_paths = [Path(p) for p in qc_paths]

    qc_files: dict[str, set[str]] = {mask_type: set() for mask_type in prefixes}
    for qc_path in qc_paths:
        for mask_type, prefix in prefixes.items():
            found = {
                p.name.removeprefix(prefix)
                for p in qc_path.iterdir()
                if p.is_file() and p.name.startswith(prefix)
            }
            qc_files[mask_type] |= found

    for mask_type, found in qc_files.items():
        prefix = prefixes[mask_type]
        print(f"{len(found)} {mask_type} mask files found (prefix: {prefix})")
    return qc_files


def _find_qc_mask(filename: str, qc_paths) -> Path | None:
    """Return the full path of a QC mask file across the given dirs, or None if not found."""
    for qc_path in qc_paths:
        candidate = Path(qc_path) / filename
        if candidate.is_file():
            return candidate
    return None

In [8]:
# ---- B6. Coverage report + summary + extras ----
DATA_MAPPING_COLUMNS = ["record_num", "mammaprint_index", "type", "path"]


def build_coverage_report(
    expected_names: set[str],
    mlflow_files: dict[str, set[str]],
    qc_files: dict[str, set[str]],
) -> pd.DataFrame:
    """Per-slide presence table across all mask types, plus an `all_present` column."""
    rows = []
    for name in sorted(expected_names):
        row = {"tiff_name": name}
        for mask_name in mlflow_files:
            row[mask_name] = name in mlflow_files[mask_name]
        for mask_type in qc_files:
            row[mask_type] = name in qc_files[mask_type]
        rows.append(row)

    mask_cols = [*mlflow_files.keys(), *qc_files.keys()]
    report = pd.DataFrame(rows)
    report["all_present"] = report[mask_cols].all(axis=1)
    return report


def coverage_summary(report: pd.DataFrame, expected_names: set[str]) -> pd.DataFrame:
    """Found/missing/total counts per mask type."""
    mask_cols = [c for c in report.columns if c not in ("tiff_name", "all_present")]
    summary = report[mask_cols].sum().to_frame("found")
    summary["missing"] = len(expected_names) - summary["found"]
    summary["total"] = len(expected_names)
    return summary


def slides_missing_masks(report: pd.DataFrame) -> pd.DataFrame:
    """Rows for slides missing at least one mask."""
    missing = report[~report["all_present"]].drop(columns=["all_present"])
    print(f"{len(missing)} slides missing at least one mask.")
    return missing


def missing_mask_data_mapping(
    df: pd.DataFrame,
    report: pd.DataFrame,
    mask_types: str | list[str],
    out_csv: str | Path | None = None,
) -> pd.DataFrame:
    """Return a data_mapping.csv-style frame of the slides MISSING the given mask type(s).

    ``mask_types`` is a mask column name from the report (e.g. "epithelium") or a list of them;
    a row is included if it is missing ANY of the requested masks. The result keeps only the
    original data_mapping columns. If ``out_csv`` is given, it is also written there.
    """
    if isinstance(mask_types, str):
        mask_types = [mask_types]

    unknown = [m for m in mask_types if m not in report.columns]
    if unknown:
        raise KeyError(f"Unknown mask type(s) {unknown}. Available: "
                       f"{[c for c in report.columns if c not in ('tiff_name', 'all_present')]}")

    # tiff_names missing at least one of the requested masks
    missing_mask = ~report[mask_types].all(axis=1)
    missing_names = set(report.loc[missing_mask, "tiff_name"])

    out = df[df["tiff_name"].isin(missing_names)]
    keep = [c for c in DATA_MAPPING_COLUMNS if c in out.columns]
    out = out[keep].reset_index(drop=True)

    print(f"{len(out)} slides missing {mask_types}.")
    if out_csv is not None:
        out.to_csv(out_csv, index=False)
        print(f"Wrote {out_csv}")
    return out


def extra_masks(
    expected_names: set[str],
    mlflow_files: dict[str, set[str]],
    qc_files: dict[str, set[str]],
) -> None:
    """Print mask files that have no matching slide in data_mapping."""
    for mask_name in mlflow_files:
        extra = mlflow_files[mask_name] - expected_names
        print(f"{len(extra)} extra {mask_name} masks with no matching slide in data_mapping:")
        if extra:
            print(sorted(extra))
        print()

    for mask_type in qc_files:
        extra = qc_files[mask_type] - expected_names
        print(f"{len(extra)} extra {mask_type} masks with no matching slide:")
        if extra:
            print(sorted(extra))

In [9]:
# ---- B5. Mask readability check (open each existing mask file) ----
def _try_open_mask(path: Path) -> str | None:
    """Try to open a mask file. Returns None on success or an error message."""
    try:
        str_path = str(path)
        if str_path.lower().endswith((".ome.tiff", ".ome.tif")):
            with TiffFile(str_path):
                pass
        else:
            with OpenSlide(str_path):
                pass
        return None
    except Exception as e:
        return f"{type(e).__name__}: {e}"


def open_masks(
    expected_names: set[str],
    mlflow_dirs: dict[str, Path],
    mlflow_files: dict[str, set[str]],
    qc_files: dict[str, set[str]],
    qc_paths=QC_MASKS_PATHS,
    prefixes=QC_MASK_PREFIXES,
) -> pd.DataFrame:
    """Open each existing mask file to verify readability. Returns a DataFrame of failures.

    Pass MLflow dirs/files from ``download_mlflow_masks`` (opening implies a prior download).
    Pass QC files from ``list_qc_masks`` (already local; may span multiple ``qc_paths``, which are
    searched in order). Any group may be an empty dict to skip it.
    """
    if isinstance(qc_paths, (str, Path)):
        qc_paths = [qc_paths]
    qc_paths = [Path(p) for p in qc_paths]

    open_errors: list[dict[str, str]] = []

    # MLflow masks (require downloaded dirs)
    for mask_name in mlflow_files:
        mask_dir = mlflow_dirs[mask_name]
        for name in sorted(expected_names & mlflow_files[mask_name]):
            err = _try_open_mask(mask_dir / name)
            if err:
                open_errors.append({"mask_type": mask_name, "tiff_name": name, "error": err})

    # QC masks (local; located across qc_paths)
    for mask_type in qc_files:
        prefix = prefixes[mask_type]
        for name in sorted(expected_names & qc_files[mask_type]):
            path = _find_qc_mask(f"{prefix}{name}", qc_paths)
            if path is None:
                open_errors.append(
                    {"mask_type": mask_type, "tiff_name": name, "error": "FileNotFound in qc_paths"}
                )
                continue
            err = _try_open_mask(path)
            if err:
                open_errors.append({"mask_type": mask_type, "tiff_name": name, "error": err})

    errors_df = pd.DataFrame(open_errors)
    if errors_df.empty:
        print("All checked mask files opened successfully.")
    else:
        print(f"{len(errors_df)} mask files failed to open.")
    return errors_df

## Visual inspection (function)

`plot_sample_masks(...)` displays N sample masks for each mask type (MLflow + QC) in a grid.
Requires downloaded MLflow dirs (from `download_mlflow_masks`).

In [10]:
# ---- B7. Visual inspection ----
THUMB_SIZE = (512, 512)


def _read_mask_thumbnail(path: Path, thumb_size=THUMB_SIZE) -> np.ndarray | None:
    """Read a mask file and return a thumbnail as a numpy array."""
    try:
        str_path = str(path)
        if str_path.lower().endswith((".ome.tiff", ".ome.tif")):
            with TiffFile(str_path) as tif:
                page = tif.pages[0]
                # Read full page then downsample via slicing (nearest-neighbor)
                img = page.asarray()
                step_y = max(1, img.shape[0] // thumb_size[1])
                step_x = max(1, img.shape[1] // thumb_size[0])
                return img[::step_y, ::step_x]
        else:
            with OpenSlide(str_path) as slide:
                thumb = slide.get_thumbnail(thumb_size)
                return np.array(thumb)
    except Exception as e:
        print(f"  Could not read {path.name}: {e}")
        return None


def plot_sample_masks(
    expected_names: set[str],
    mlflow_dirs: dict[str, Path],
    mlflow_files: dict[str, set[str]],
    qc_files: dict[str, set[str]],
    n_samples: int = 5,
    qc_paths=QC_MASKS_PATHS,
    prefixes=QC_MASK_PREFIXES,
) -> None:
    """Display n_samples sample masks per type (MLflow + QC) in a grid.

    QC masks are located across ``qc_paths`` (searched in order), so masks spread over several
    directories are handled as if they lived in a single location.
    """
    if isinstance(qc_paths, (str, Path)):
        qc_paths = [qc_paths]
    qc_paths = [Path(p) for p in qc_paths]

    # Collect mask paths: list of (mask_type, [paths])
    all_mask_types: list[tuple[str, list[Path]]] = []
    sample_names = sorted(expected_names)[:n_samples]

    for mask_name in mlflow_files:
        paths = [mlflow_dirs[mask_name] / name for name in sample_names if name in mlflow_files[mask_name]]
        all_mask_types.append((mask_name, paths[:n_samples]))

    for mask_type in qc_files:
        prefix = prefixes[mask_type]
        paths = [
            found
            for name in sample_names
            if name in qc_files[mask_type]
            and (found := _find_qc_mask(f"{prefix}{name}", qc_paths)) is not None
        ]
        all_mask_types.append((mask_type, paths[:n_samples]))

    # Plot grid: rows = mask types, cols = samples
    n_rows = len(all_mask_types)
    n_cols = n_samples
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3 * n_rows))
    if n_rows == 1:
        axes = [axes]

    for row_idx, (mask_type, paths) in enumerate(all_mask_types):
        for col_idx in range(n_cols):
            ax = axes[row_idx][col_idx]
            ax.set_xticks([])
            ax.set_yticks([])
            if col_idx == 0:
                ax.set_ylabel(mask_type, fontsize=12, fontweight="bold")
            if col_idx < len(paths):
                thumb = _read_mask_thumbnail(paths[col_idx])
                if thumb is not None:
                    ax.imshow(thumb, cmap="gray")
                    ax.set_title(paths[col_idx].name, fontsize=7)
                else:
                    ax.text(0.5, 0.5, "read error", ha="center", va="center", transform=ax.transAxes)
            else:
                ax.text(0.5, 0.5, "N/A", ha="center", va="center", transform=ax.transAxes, color="gray")

    fig.suptitle(f"Sample masks ({n_samples} per type)", fontsize=14, fontweight="bold")
    fig.tight_layout()
    plt.show()

## Tiled dataset validation (functions)

`download_tiled_datasets(...)` downloads `slides.parquet` / `tiles.parquet` per dataset.
`validate_tiled_datasets(df, tiled_data)` checks:
1. Every slide in `data_mapping.csv` has a row in `slides.parquet` (and vice versa)
2. Every slide in `slides.parquet` has at least one tile in `tiles.parquet`
3. Every `slide_id` in `tiles.parquet` maps to a slide in `slides.parquet`

In [11]:
# ---- B8a. Tiled datasets ----
def download_tiled_datasets(tiled=TILED_DATASETS) -> dict[str, tuple[pd.DataFrame, pd.DataFrame]]:
    """Download slides.parquet / tiles.parquet per dataset. Returns {name: (slides, tiles)}."""
    tiled_data: dict[str, tuple[pd.DataFrame, pd.DataFrame]] = {}
    for ds_name, uri in tiled.items():
        local_dir = Path(download_artifacts(uri))
        slides = pd.read_parquet(local_dir / "slides.parquet")
        tiles = pd.read_parquet(local_dir / "tiles.parquet")
        tiled_data[ds_name] = (slides, tiles)
        print(f"{ds_name}: {len(slides)} slides, {len(tiles)} tiles")
    return tiled_data


def validate_tiled_datasets(df: pd.DataFrame, tiled_data: dict) -> None:
    """Check slides.parquet <-> data_mapping and tiles.parquet <-> slides.parquet consistency."""
    expected_stems = {Path(p).stem for p in df["path"]}

    for ds_name, (slides, tiles) in tiled_data.items():
        print(f"=== {ds_name} ===\n")

        # 1. slides.parquet path stem <-> data_mapping path stem
        slide_stems = {Path(p).stem for p in slides["path"]}
        missing_from_dataset = expected_stems - slide_stems
        extra_in_dataset = slide_stems - expected_stems

        print(f"Slides in data_mapping: {len(expected_stems)}")
        print(f"Slides in dataset:      {len(slide_stems)}")
        print(f"Missing from dataset:   {len(missing_from_dataset)}")
        print(f"Extra in dataset:       {len(extra_in_dataset)}")

        if missing_from_dataset:
            print(f"\nSlides in data_mapping but NOT in {ds_name}:")
            for p in sorted(missing_from_dataset):
                print(f"  {p}")

        if extra_in_dataset:
            print(f"\nSlides in {ds_name} but NOT in data_mapping:")
            for p in sorted(extra_in_dataset):
                print(f"  {p}")

        # 2. Every slide has at least one tile
        slide_ids = set(slides["id"])
        slide_ids_in_tiles = set(tiles["slide_id"])

        slides_without_tiles = slide_ids - slide_ids_in_tiles
        print(f"\nSlides without any tiles: {len(slides_without_tiles)}")
        if slides_without_tiles:
            for s in sorted(slides_without_tiles):
                print(f"  {s}")

        # 3. No orphan tiles (tile references a slide not in slides.parquet)
        orphan_tile_ids = slide_ids_in_tiles - slide_ids
        print(f"Orphan tile slide_ids:    {len(orphan_tile_ids)}")
        if orphan_tile_ids:
            for sid in sorted(orphan_tile_ids):
                count = int((tiles["slide_id"] == sid).sum())
                print(f"  slide_id={sid}: {count} tiles")

        print()

## Embeddings validation (functions)

`list_embeddings(df)` checks coverage using `list_artifacts` (no download).
`validate_embeddings(df)` downloads the artifact dir and checks the same, for parity with the
download+open mask mode. Both verify every slide in `data_mapping.csv` has a matching `.parquet` file.

In [12]:
# ---- B8b. Embeddings: two modes ----
# Embedding parquet files are named as "{record_num}.parquet" (e.g. "2023/00835.parquet"),
# i.e. the stem matches the data_mapping `path` column.
def _report_embeddings(emb_name: str, emb_files: set[str], expected_record_nums: set[str]) -> None:
    print(f"=== {emb_name} ===\n")
    print(f"Embedding files found:  {len(emb_files)}")
    print(f"Slides in data_mapping: {len(expected_record_nums)}")

    missing = expected_record_nums - emb_files
    extra = emb_files - expected_record_nums

    print(f"Missing embeddings:     {len(missing)}")
    if missing:
        for s in sorted(missing):
            print(f"  {s}")

    print(f"Extra embeddings:       {len(extra)}")
    if extra:
        for s in sorted(extra):
            print(f"  {s}")
    print()


def list_embeddings(df: pd.DataFrame, embeddings=EMBEDDINGS) -> None:
    """LIST ONLY: check embedding coverage via list_artifacts, without downloading."""
    expected_record_nums = set(df["path"])
    for emb_name, uri in embeddings.items():
        emb_files = {
            Path(info.path).stem
            for info in list_artifacts(artifact_uri=uri)
            if not info.is_dir and info.path.endswith(".parquet")
        }
        _report_embeddings(emb_name, emb_files, expected_record_nums)


def validate_embeddings(df: pd.DataFrame, embeddings=EMBEDDINGS) -> None:
    """DOWNLOAD: fetch the embeddings dir and check coverage against data_mapping."""
    expected_record_nums = set(df["path"])
    for emb_name, uri in embeddings.items():
        emb_dir = Path(download_artifacts(uri))
        emb_files = {p.stem for p in emb_dir.iterdir() if p.is_file() and p.suffix == ".parquet"}
        _report_embeddings(emb_name, emb_files, expected_record_nums)

# Calls

Run the cells below to execute individual checks. **Comment out whatever you don't need.**

The cheap, network-light path is left uncommented by default (list-only mask enumeration + coverage
report). The download / open / visual / tiled / embeddings cells are commented out — uncomment them
when you want to actually fetch and open the files to verify everything works end-to-end.

In [13]:
# --- Load data mapping (required by everything below) ---
df, expected_names = load_data_mapping()
df.head()

Loaded 2042 slides from /mnt/projects/mammaprint/data_mapping.csv
Columns: ['record_num', 'mammaprint_index', 'type', 'path', 'split', 'tiff_name']
2042 unique slide tiff names expected


,record_num,mammaprint_index,type,path,split,tiff_name
0,2023/00835,-0.158,b luminal,/mnt/data/MOU/breast/mammaprint/P2023_0835,test,P2023_0835.tiff
1,2023/00836,0.058,a luminal,/mnt/data/MOU/breast/mammaprint/P2023_0836,train,P2023_0836.tiff
2,2023/00837,-0.039,b luminal,/mnt/data/MOU/breast/mammaprint/P2023_0837,train,P2023_0837.tiff
3,2023/00838,0.065,a luminal,/mnt/data/MOU/breast/mammaprint/P2023_0838,test,P2023_0838.tiff
4,2023/00839,-0.429,b luminal,/mnt/data/MOU/breast/mammaprint/P2023_0839,test,P2023_0839.tiff


In [14]:
# --- Slides open check (verifies every slide file in data_mapping opens) ---
# slide_errors = check_slides_open(df)
# display(slide_errors)

In [15]:
# --- MLflow masks: LIST ONLY (no download) ---
mlflow_files = list_mlflow_masks()

# --- MLflow masks: DOWNLOAD + OPEN (uncomment to actually fetch & verify readability) ---
# mlflow_dirs, mlflow_files = download_mlflow_masks()
# mlflow_open_errors = open_masks(expected_names, mlflow_dirs, mlflow_files, {})
# display(mlflow_open_errors)

2042 tissue mask files listed (no download) at mlflow-artifacts:/3/1d67612f123e46f6a3c6183257106512/artifacts/tissue_masks
310 epithelium mask files listed (no download) at mlflow-artifacts:/3/0176fb9485de49699b1e16f362cff5fa/artifacts/epithelium_masks


In [16]:
# --- QC masks: LIST ONLY (local ls, no open) ---
qc_files = list_qc_masks()

# --- QC masks: OPEN (uncomment to verify each local QC mask is readable) ---
# qc_open_errors = open_masks(expected_names, {}, {}, qc_files)
# display(qc_open_errors)

1800 blur mask files found (prefix: Piqe_piqe_median_activity_mask_)
1800 folding mask files found (prefix: FoldingFunction_folding_test_)
1800 residual mask files found (prefix: ResidualArtifactsAndCoverage_coverage_mask_)


In [17]:
# --- Coverage report (per-slide presence across all mask types) ---
report = build_coverage_report(expected_names, mlflow_files, qc_files)
display(report)

# --- Summary / missing / extras (uncomment as needed) ---
# display(coverage_summary(report, expected_names))
# display(slides_missing_masks(report))
# extra_masks(expected_names, mlflow_files, qc_files)

,tiff_name,tissue,epithelium,blur,folding,residual,all_present
0,2024_02851-1.tiff,True,True,True,True,True,True
1,2024_02865.tiff,True,False,True,True,True,False
2,2024_02890.tiff,True,False,True,True,True,False
3,P2023_01000.tiff,True,False,True,True,True,False
4,P2023_01001.tiff,True,False,True,True,True,False
...,...,...,...,...,...,...,...
2037,P2026_00970.tiff,True,False,False,False,False,False
2038,P2026_00973.tiff,True,False,False,False,False,False
2039,P2026_00974.tiff,True,False,False,False,False,False
2040,P2026_00976.tiff,True,False,False,False,False,False


In [18]:
# --- Export a data_mapping.csv-style file of slides MISSING a chosen mask ---
# Set mask_types to the mask column you care about, e.g. "epithelium", "tissue",
# "blur", "folding", "residual" (or a list of several -> missing ANY of them).
missing_epithelium = missing_mask_data_mapping(
    df, report, mask_types=["blur", "folding", "residual"], out_csv="data_mapping_missing_qc.csv"
)
display(missing_epithelium)

243 slides missing ['blur', 'folding', 'residual'].
Wrote data_mapping_missing_qc.csv


,record_num,mammaprint_index,type,path
0,2024/04733,-0.122,b luminal,/mnt/data/MOU/breast/mammaprint/P2024_04733
1,2025/00161,0.263,a luminal,/mnt/data/MOU/breast/mammaprint/P2025_00161-2
2,2025/04084,-0.215,b luminal,/mnt/data/MOU/breast/mammaprint/P2025_04084
3,2025/04262,-0.023,b luminal,/mnt/data/MOU/breast/mammaprint/P2025_04262
4,2025/04267,-0.309,b luminal,/mnt/data/MOU/breast/mammaprint/P2025_04267
...,...,...,...,...
238,2026/00970,-0.218,b luminal,/mnt/data/MOU/breast/mammaprint/P2026_00970
239,2026/00973,-0.141,b luminal,/mnt/data/MOU/breast/mammaprint/P2026_00973
240,2026/00974,0.192,a luminal,/mnt/data/MOU/breast/mammaprint/P2026_00974
241,2026/00976,0.396,a luminal,/mnt/data/MOU/breast/mammaprint/P2026_00976


In [19]:
# --- Visual grid: sample masks per type (needs downloaded MLflow dirs; uncomment to run) ---
# mlflow_dirs, mlflow_files = download_mlflow_masks()
# plot_sample_masks(expected_names, mlflow_dirs, mlflow_files, qc_files)

In [20]:
# --- Tiled datasets (downloads slides/tiles parquet; uncomment to run) ---
# tiled_data = download_tiled_datasets()
# validate_tiled_datasets(df, tiled_data)

In [21]:
# --- Embeddings ---
# list_embeddings(df)        # LIST ONLY (no download)
# validate_embeddings(df)    # DOWNLOAD + check coverage